# Combining the Exercise Datasets
### Kaya Daylor

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/BA 878/BA878_class_project/Exercise Data/' # Can change to put your own file path here

In [ ]:
# Getting the Data
active_minutes = pd.read_csv(file_path + 'active_minutes.csv').drop_duplicates()
vo2_max = pd.read_csv(file_path + 'demographic_vo2_max.csv').drop_duplicates()
exercise = pd.read_csv(file_path + 'exercise.csv').drop_duplicates()
steps = pd.read_csv(file_path + 'steps.csv').drop_duplicates()
calories = pd.read_csv(file_path + 'calories.csv').drop_duplicates()

daily_steps = steps.groupby(['id', 'study_interval', 'is_weekend', 'day_in_study'], as_index=False)['steps'].sum()
daily_calories = calories.groupby(['id', 'study_interval', 'is_weekend', 'day_in_study'], as_index=False)['calories'].sum()

subcategory_map = {
    "Walk": "Cardio",
    "Run": "Cardio",
    "Outdoor Bike": "Cardio",
    "Bike": "Cardio",
    "Treadmill": "Cardio",
    "Elliptical": "Cardio",
    "Aerobic Workout": "Cardio",
    "Swim": "Cardio",
    "Hike": "Cardio",

    "Weights": "Strength",
    "Bootcamp": "Strength",
    "Workout": "Strength",

    "Sport": "Sports",
    "Tennis": "Sports",
    "Golf": "Sports",
    "Yoga": "Sports" }

exercise["activitycategory"] = exercise["activityname"].map(subcategory_map)

# Dropping Columns + Renaming Columns
vo2_max = vo2_max.drop(columns = ['demographic_vo2_max_error', 'filtered_demographic_vo2_max', 'filtered_demographic_vo2_max_error'], axis = 1)
exercise = exercise.rename(columns = {'start_day_in_study' : 'day_in_study', 'steps' : 'steps_exercise', 'calories' : 'calories_exercise'})
exercise = exercise.drop(columns = ['last_modified_day_in_study', 'last_modified_timestamp', 'original_start_day_in_study', 'original_start_timestamp', 'originalduration', 'activitytypeid', 'logtype', 'manualvaluesspecified', 'elevationgain', 'hasgps', 'shouldfetchdetails', 'hasactivezoneminutes', 'activeduration', 'heartratezones', 'activezoneminutes', 'start_timestamp', 'activityname', 'activitylevel'], axis = 1).drop_duplicates()

exercise = (
    exercise.groupby(['id', 'study_interval', 'is_weekend', 'day_in_study'])
      .agg({
          'averageheartrate': 'mean',
          'calories_exercise': 'sum',
          'duration': 'sum',
          'steps_exercise': 'sum',
          'activitycategory': lambda x: list(x)
      }).reset_index())

exercise_categories = ['Cardio', 'Sports', 'Strength']

# For each category, count occurrences in the activitycategory list
for cat in exercise_categories:
    exercise[cat] = exercise['activitycategory'].apply(lambda x: x.count(cat))


In [ ]:
print(vo2_max.columns)
print(exercise.columns)
print(daily_steps.columns)
print(daily_calories.columns)

In [ ]:
# Merge Exercise Data
daily_merged = pd.merge(daily_calories, daily_steps, on=['id', 'day_in_study', 'study_interval'], how='left')
#daily_merged = pd.merge(daily_merged, vo2_max, on=['id', 'day_in_study', 'study_interval', 'is_weekend'], how='inner')
daily_merged = pd.merge(daily_merged, exercise, on=['id', 'day_in_study', 'study_interval'], how='left').drop(columns = ['activitycategory', 'is_weekend_x', 'is_weekend_y'], axis = 1)


In [ ]:
print(daily_merged.columns)
print(daily_merged.shape)
print(daily_merged.tail())

In [ ]:
def check_unique(df, name):
    is_unique = not df.duplicated(["id","study_interval", "day_in_study"]).any()
    dup_count = df.duplicated(["id","study_interval", "day_in_study"], keep=False).sum()
    print(f"{name}: {'✅ Unique' if is_unique else '⚠️ Not Unique'}, it has {dup_count} duplicates")
check_unique(exercise, "")

## ARCHIVE

In [ ]:
print("Active Mins", active_minutes.info())
print("VO2 Max", vo2_max.info())
print("Exercise", exercise.info())
print("Steps", steps.info())
print("Calories", calories.info())

### daily_merged EDA

In [ ]:
print("Shape: ", daily_merged.shape) # (3531, 12)
print(daily_merged[['id', 'day_in_study', 'is_weekend']].nunique())

In [ ]:
print(daily_merged.describe())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = daily_merged.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()
